# OncoEvidence, Part 8 of 10: Popularity-deconfounded validation

> **Reproducible template.** Outputs are cleared in the repo. Run this notebook top-to-bottom on a GPU (e.g. Colab) to regenerate them; set `FAST_MODE=False` for the full (paper) run. The headline numbers it produces are also recorded inline below and in `results/*.json`.

Does the shortlist hold up against real trials, once popularity is removed? (Extension.)

This is one of ten notebooks. Parts 1 to 3 build the core pipeline (foundations, mechanisms, evidence and recovery), Part 4 is the full self-contained notebook, and Parts 5 to 10 are advanced extensions (prospective validation, mechanism faithfulness, prospective named cases, popularity-deconfounded trial validation, the counterfactual mechanism stress test, and independent functional-genomics corroboration). Each part runs on its own and is Colab-ready; set the runtime to GPU under Runtime, Change runtime type. The toolbox cells (L1, L2, and so on) are the same building blocks in every part and match the full Part 4 notebook, so as we work through the track we are assembling the complete system one piece at a time.

Everything here is hypothesis-generating research, not medical advice.

What Part 8 covers:

- The confound: a naive ClinicalTrials.gov check is driven by popularity on two sides (popular drugs and popular cancers are trialed for everything).
- The fix: fit an additive drug-effect plus cancer-effect model to the scores, subtract both, and test whether the drug-by-cancer interaction residual still predicts a real trial.
- The result, reported honestly: the within-drug signal looks strong (recorded 0.821) but vanishes once cancer popularity is also removed (interaction 0.475, no signal). The apparent effect was popularity, not pairwise specificity, a clean negative that contrasts with the prospective wins in Parts 5 and 7.

This part trains a deployment GNN and makes live ClinicalTrials.gov calls (cached), so a GPU helps and the first run touches the network.

## 0. Configuration

This cell sets how heavy the run is. There are two switches.

`FAST_MODE` trades speed for fidelity. Left at True it runs a quick demo: one seed, fewer epochs, and cheap hashing features instead of learned embeddings. That is enough to watch every result take shape, and it finishes in a few minutes. Set it to False to reproduce the published numbers (5 seeds, SentenceTransformer embeddings, XGBoost tuning), which takes a few hours on a GPU.

`RUN_LLM` turns the LLM reviewer on or off. With it on we need an OpenAI-compatible API key. With it off, the literature check still runs using a transparent keyword grader, so nothing requires a key.

The remaining values are budgets (epochs, seed counts, sample sizes) that follow from `FAST_MODE`.

In [ ]:
# Master switches
FAST_MODE = False  # True: quick demo (minutes). False: full published reproduction (hours).
RUN_LLM   = False  # True: call a real LLM reviewer (needs an API key below).

import os

# LLM credentials (only used when RUN_LLM=True)
# The LLM client is OpenAI-compatible. The default base URL points at OpenRouter,
# which proxies many models; for the official OpenAI API use https://api.openai.com/v1.
os.environ.setdefault("ONCO_LLM_API_KEY", "")               # <-- paste the API key here to enable the LLM
ONCO_LLM_BASE_URL = "https://openrouter.ai/api/v1"          # OpenAI: https://api.openai.com/v1
ONCO_LLM_MODEL    = "openai/gpt-4o-mini"                    # a small, cheap, capable judge model
if RUN_LLM and os.environ["ONCO_LLM_API_KEY"]:
    os.environ["ONCO_LLM_BASE_URL"] = ONCO_LLM_BASE_URL
    os.environ["ONCO_LLM_MODEL"]    = ONCO_LLM_MODEL

# Training / evaluation budgets (auto-scale with FAST_MODE)
# In fast mode we use a single seed and short training; in full mode we average over
# 5 seeds and train longer, which is what the paper's numbers were produced with.
SEEDS              = [0] if FAST_MODE else [0, 1, 2, 42, 123]  # random seeds we average over
ABLATION_SEEDS     = [0] if FAST_MODE else [0, 1, 2]           # seeds for the (expensive) ablations
GNN_EPOCHS         = 12 if FAST_MODE else 50                   # GNN training epochs (early-stopped)
MLP_EPOCHS         = 50 if FAST_MODE else 200                  # MLP baseline epochs
KGE_EPOCHS         = 80 if FAST_MODE else 300                  # knowledge-graph-embedding epochs
PATIENCE           = 6 if FAST_MODE else 10                    # early-stopping patience (val AUROC)
XGB_TUNE           = (not FAST_MODE)                           # tune XGBoost only in full mode
XGB_TRIALS         = 5 if FAST_MODE else 8                     # Optuna trials when tuning
XGB_ESTIMATORS     = 200 if FAST_MODE else 400                 # XGBoost trees (upper bound; early-stopped)
USE_FALLBACK_FEATS = FAST_MODE                                 # hashing feats (instant) vs ST embeddings (slower)
N_SEP              = 120 if FAST_MODE else 400                 # pairs per group in the separation test
REC_EPOCHS         = 12 if FAST_MODE else 60                   # mechanism-recovery training epochs
REC_SEEDS          = [0] if FAST_MODE else [0, 1, 2]           # seeds for mechanism recovery

print(f"FAST_MODE={FAST_MODE}  RUN_LLM={RUN_LLM}")
print(f"seeds={SEEDS}  gnn_epochs={GNN_EPOCHS}  xgb_tune={XGB_TUNE}  fallback_features={USE_FALLBACK_FEATS}")

## 1. Install dependencies

Colab already ships PyTorch and the usual scientific stack, so we only add the extras this notebook needs:

- `torch-geometric`: the graph neural network layers.
- `xgboost`: gradient-boosted trees, the strongest tabular baseline here.
- `optuna`: hyper-parameter search for XGBoost, used only in full mode.
- `sentence-transformers` and `transformers`: turn node names like "imatinib" into embedding vectors.
- `scikit-learn`, `pandas`, `numpy`, `scipy`: metrics, tables, statistics.
- `matplotlib`, `seaborn`: the plots.
- `requests`: calls to the Europe PMC, DrugMechDB, and mygene web APIs.
- `pyyaml`, `tqdm`: DrugMechDB parsing and progress bars.

In [ ]:
# Install only the extras Colab doesn't ship with. (Safe to re-run; pip is idempotent.)
%pip install -q torch-geometric xgboost optuna "sentence-transformers>=2.2.0" "transformers>=4.40,<5" scikit-learn pandas numpy scipy matplotlib seaborn requests tqdm pyyaml
print("Dependencies installed.")

In [ ]:
# Pick the compute device. A GPU makes GNN training ~10-50x faster than CPU.
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    # Not fatal - everything still runs on CPU, just slowly. Prefer enabling a GPU runtime.
    print("WARNING: no GPU detected. Use Runtime > Change runtime type > GPU for reasonable speed.")

## Toolbox: run these once

The cells below only define the functions this part needs. They run fast and print almost nothing. They keep the same L-numbers as the full notebook so the pieces line up across the track. We can read them to see how things work, or just run them in order and move on to the workflow.

Nothing here imports from a project package. Each part stands alone.

### L1. Paths, schema constants, oncology keywords

This sets up a few things the rest of the notebook relies on:

- The cache folders (`data/`, `models/`, `results/`, `figures/`) so repeated runs reuse work instead of redoing it.
- The PrimeKG download URL and the cache filenames.
- Schema constants. PrimeKG is a heterogeneous graph with many node and edge types. We name the ones we use: `drug`, `disease`, `gene_protein`, and the three drug-disease therapeutic relations (`indication`, `contraindication`, `off-label use`). Those therapeutic edges are the labels we predict, so they get careful handling later to avoid leakage.
- `ONCOLOGY_KEYWORDS`, a small word list for flagging which diseases are cancers, so evaluation can be restricted to oncology.

In [ ]:
from pathlib import Path

# Local cache/output directories. On Colab these live under /content and reset per session.
DATA_DIR    = Path("data")     # raw PrimeKG csv + built graph + small json caches
MODELS_DIR  = Path("models")   # feature caches + LLM response cache
RESULTS_DIR = Path("results")  # metric JSONs
FIGURES_DIR = Path("figures")  # saved plots
for _d in (DATA_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# PrimeKG: a large public biomedical knowledge graph (~8M edges) hosted on Harvard Dataverse.
PRIMEKG_KG_CSV = DATA_DIR / "kg.csv"                                    # raw edge list (~980 MB)
PRIMEKG_KG_URL = "https://dataverse.harvard.edu/api/access/datafile/6180620"
HETERODATA_PT  = DATA_DIR / "primekg_hetero.pt"                         # parsed PyG graph (cached)
FEATURE_CACHE  = MODELS_DIR / "primekg_text_features.pt"               # node embeddings (cached)
LLM_CACHE_DIR  = MODELS_DIR / "llm_cache"                              # cache LLM calls (save $ + time)
UNIPROT_CACHE  = DATA_DIR / "uniprot2symbol.json"                     # UniProt->gene-symbol map (cached)

# The drug<->disease "therapeutic" relations. `indication` (drug treats disease) is our
# main prediction target; all three are stripped from the message-passing graph to
# prevent the model from "seeing the answer" (label leakage).
INDICATION_REL = "indication"; CONTRAINDICATION_REL = "contraindication"; OFFLABEL_REL = "off-label use"
THERAPEUTIC_RELS = (INDICATION_REL, CONTRAINDICATION_REL, OFFLABEL_REL)

# Node-type names exactly as they appear in PrimeKG after normalization.
DRUG_TYPE = "drug"; DISEASE_TYPE = "disease"; GENE_TYPE = "gene_protein"

# SentenceTransformer model used to embed node names into vectors (full mode).
TEXT_MODEL = "all-MiniLM-L6-v2"

# Substring rules to flag a disease as cancer. Crude but effective on PrimeKG names.
ONCOLOGY_KEYWORDS = (
    "cancer","carcinoma","sarcoma","neoplasm","tumor","tumour","leukemia","leukaemia",
    "lymphoma","melanoma","glioma","glioblastoma","myeloma","blastoma","adenoma",
    "malignant","metastat","oncocytoma","mesothelioma","astrocytoma","teratoma","carcinosarcoma",
)
print("L1 constants ready")

### L2. Download PrimeKG and build the graph

PrimeKG comes as a flat CSV edge list, one row per edge: `(x_type, x_id, x_name, relation, y_type, y_id, y_name)`. We turn it into a PyTorch Geometric `HeteroData` object, which is what the GNN reads.

The build has three steps:
1. Register nodes. Give every unique `(type, id)` a contiguous integer index within its type, and keep its name. Models work on the integer indices.
2. Group edges. Collect rows into `(source_type, relation, dest_type)` edge types, each stored as a `[2, num_edges]` index tensor.
3. Flag oncology diseases with the keyword list and store the result as a boolean mask.

The parsed graph is cached to `primekg_hetero.pt`, so the slow part only runs once.

In [ ]:
import sys, re, urllib.request
from collections import defaultdict
import pandas as pd
import torch
from torch_geometric.data import HeteroData

def download_primekg(dest=PRIMEKG_KG_CSV, force=False):
    '''Download the PrimeKG edge list if we don't already have it (~980 MB, one-time).'''
    dest = Path(dest)
    if dest.exists() and not force and dest.stat().st_size > 0:
        print(f"PrimeKG already present: {dest} ({dest.stat().st_size/1e6:.0f} MB)"); return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading PrimeKG -> {dest} (~980 MB, this can take a few minutes)")
    def _progress(block_num, block_size, total_size):  # simple percent printout
        if total_size > 0:
            sys.stdout.write(f"\r  {min(100.0, block_num*block_size*100.0/total_size):5.1f}%"); sys.stdout.flush()
    urllib.request.urlretrieve(PRIMEKG_KG_URL, dest, _progress); sys.stdout.write("\n")
    print(f"Saved {dest} ({dest.stat().st_size/1e6:.0f} MB)"); return dest

# Normalize free-text type/relation strings into clean snake_case tokens (e.g. "Drug Protein"
# -> "drug_protein"), so the graph schema is consistent regardless of source capitalization.
def _norm_type(t): return re.sub(r"[^0-9a-zA-Z]+", "_", str(t).strip().lower()).strip("_")
def _norm_rel(r):  return re.sub(r"[^0-9a-zA-Z]+", "_", str(r).strip().lower()).strip("_")
def _is_oncology(name): return any(k in str(name).lower() for k in ONCOLOGY_KEYWORDS)

def build_hetero_from_primekg(kg_csv=PRIMEKG_KG_CSV, output_path=HETERODATA_PT, save=True):
    '''Parse the PrimeKG CSV into a PyG HeteroData graph and (optionally) cache it.'''
    kg_csv = Path(kg_csv)
    if not kg_csv.exists():
        raise FileNotFoundError(f"{kg_csv} not found; run download_primekg() first.")
    print(f"Loading {kg_csv} ...")
    df = pd.read_csv(kg_csv, dtype=str, low_memory=False)   # read everything as strings; ids are mixed
    print(f"  {len(df):,} raw edges")
    # Normalize type/relation columns up front (vectorized, fast).
    df["x_type_n"] = df["x_type"].map(_norm_type); df["y_type_n"] = df["y_type"].map(_norm_type)
    df["rel_n"] = df["relation"].map(_norm_rel)

    # --- Step 1: assign each (type, original_id) a contiguous per-type integer index ---
    node_id_to_idx = defaultdict(dict)    # type -> {original_id: int_index}
    node_names = defaultdict(list)        # type -> [name per index]
    node_ids = defaultdict(list)          # type -> [original_id per index]
    def _register(t, i, nm):
        m = node_id_to_idx[t]
        if i not in m:                    # first time we see this node id
            m[i] = len(m)                 # next free index for this type
            node_names[t].append("" if nm is None else str(nm))
            node_ids[t].append(str(i))
        return m[i]
    # A node can appear on either side of an edge, so scan both the x- and y-columns.
    for side in ("x", "y"):
        sub = df[[f"{side}_type_n", f"{side}_id", f"{side}_name"]].drop_duplicates()
        for t, i, nm in sub.itertuples(index=False):
            _register(t, i, nm)

    data = HeteroData()
    # Attach per-type metadata (count, original ids, names) to the graph.
    for nt, m in node_id_to_idx.items():
        data[nt].num_nodes = len(m); data[nt].node_ids = node_ids[nt]; data[nt].node_names = node_names[nt]

    # --- Step 2: group edges into (src_type, relation, dst_type) buckets ---
    buckets = defaultdict(list)
    for x_t, x_id, y_t, y_id, rel in df[["x_type_n","x_id","y_type_n","y_id","rel_n"]].itertuples(index=False):
        buckets[(x_t, rel, y_t)].append((node_id_to_idx[x_t][x_id], node_id_to_idx[y_t][y_id]))
    for et, pairs in buckets.items():
        # edge_index has shape [2, num_edges]: row 0 = source indices, row 1 = dest indices.
        data[et].edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()

    # --- Step 3: mark which diseases are cancers (used to restrict oncology evaluation) ---
    if DISEASE_TYPE in data.node_types:
        names = data[DISEASE_TYPE].node_names
        data[DISEASE_TYPE].is_oncology = torch.tensor([_is_oncology(n) for n in names], dtype=torch.bool)
        print(f"  oncology diseases flagged: {int(data[DISEASE_TYPE].is_oncology.sum())} / {len(names)}")

    total_nodes = sum(int(data[t].num_nodes) for t in data.node_types)
    total_edges = sum(int(data[e].edge_index.size(1)) for e in data.edge_types)
    print(f"  {len(data.node_types)} node types ({total_nodes:,} nodes), "
          f"{len(data.edge_types)} edge types ({total_edges:,} edges)")
    if save:
        torch.save(data, Path(output_path)); print(f"Saved parsed graph -> {output_path}")
    return data
print("L2 ready")

### L3. Node features, loader, and summary

Every node needs a feature vector. We build features from node names (for example the drug "imatinib" or the gene "BCR"):

- In full mode, a SentenceTransformer maps each name to a dense embedding, so related names sit close together.
- In fast mode or offline, a deterministic hashing of the name's character n-grams. It needs no model and no GPU, and it is good enough for a demo.

The cell also defines `load_primekg`, which builds or loads the cached graph, attaches features, and finds the indication and contraindication target edge types, and `graph_summary`, which prints a short description of whatever graph is loaded. Features are cached so they are not recomputed.

In [ ]:
import hashlib
import numpy as np

DEFAULT_HASH_DIM = 256   # dimensionality of the fallback hashing features

def _node_texts(data, nt):
    '''Build a short descriptive string per node, e.g. 'drug: imatinib'.'''
    store = data[nt]; names = getattr(store, "node_names", None); n = int(store.num_nodes)
    pretty = nt.replace("_", " ")
    if names is None:
        return [f"{pretty} {i}" for i in range(n)]
    out = []
    for i in range(n):
        raw = "" if (i >= len(names) or names[i] is None) else str(names[i]).strip()
        out.append(f"{pretty}: {raw}" if raw else f"{pretty} {i}")
    return out

def _hash_embed(texts, dim=DEFAULT_HASH_DIM):
    '''Deterministic 'feature hashing' embedding: map char n-grams into a fixed vector.

    This needs no model and no network. Each token and 3-gram is hashed to a bucket and
    adds +/-1; the vector is L2-normalized. Names sharing substrings get similar vectors.
    '''
    vecs = np.zeros((len(texts), dim), dtype=np.float32)
    for row, text in enumerate(texts):
        toks = re.findall(r"[a-z0-9]+", text.lower()); grams = list(toks)
        for tok in toks:                                   # add character 3-grams of each token
            padded = f"#{tok}#"; grams.extend(padded[i:i+3] for i in range(len(padded)-2))
        for g in grams:
            h = int(hashlib.md5(g.encode()).hexdigest(), 16)
            vecs[row, h % dim] += 1.0 if (h // dim) % 2 == 0 else -1.0   # signed hashing reduces collisions
    norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
    return vecs / norms

def _st_embed(texts_by_type, model_name):
    '''Embed node names with a SentenceTransformer; return None to trigger the hashing fallback.'''
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        print(f"  [features] sentence-transformers unavailable ({exc}); using hashing fallback"); return None
    try:
        model = SentenceTransformer(model_name); out = {}
        for nt, texts in texts_by_type.items():
            out[nt] = model.encode(texts, batch_size=512, normalize_embeddings=True,
                                   show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        return out
    except Exception as exc:
        print(f"  [features] SentenceTransformer failed ({exc}); using hashing fallback"); return None

def build_text_features(data, cache_path=FEATURE_CACHE, model_name=TEXT_MODEL, force_fallback=False):
    '''Attach an `.x` feature matrix to every node type, using cache when available.'''
    cache_path = Path(cache_path) if cache_path is not None else None
    # Keep hashing-features in a separate cache file so the two modes never collide.
    if cache_path is not None and force_fallback:
        cache_path = cache_path.with_name(cache_path.stem + "_hash" + cache_path.suffix)
    # Reuse a cache only if its row counts match the current graph exactly.
    if cache_path is not None and cache_path.exists():
        cached = torch.load(cache_path, weights_only=False)
        if all(nt in cached and cached[nt].shape[0] == int(data[nt].num_nodes) for nt in data.node_types):
            for nt in data.node_types: data[nt].x = cached[nt].float()
            print(f"  [features] loaded from cache (dim={next(iter(cached.values())).shape[1]})"); return data
    texts = {nt: _node_texts(data, nt) for nt in data.node_types}
    emb = None if force_fallback else _st_embed(texts, model_name)
    if emb is None:                                         # offline / fast path
        emb = {nt: _hash_embed(t) for nt, t in texts.items()}; src = f"hashing (dim={DEFAULT_HASH_DIM})"
    else:
        src = f"{model_name} (dim={next(iter(emb.values())).shape[1]})"
    tensors = {}
    for nt in data.node_types:
        x = torch.from_numpy(np.ascontiguousarray(emb[nt])).float(); data[nt].x = x; tensors[nt] = x
    print(f"  [features] built via {src}")
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True); torch.save(tensors, cache_path)
    return data

def _find_target_edge(data, relation):
    '''Find the (drug, relation, disease) edge type for a therapeutic relation, drug-first.'''
    rel_n = _norm_rel(relation); cands = []
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r == rel_n: cands.append(et)
    if not cands: return None
    for et in cands:
        if et[0] == DRUG_TYPE: return et    # prefer the drug->disease orientation
    return cands[0]

def load_primekg(pt_path=HETERODATA_PT, with_features=True, force_fallback_features=False, build_if_missing=True):
    '''Top-level loader: build-or-load the graph, attach features, return target edge types.'''
    pt_path = Path(pt_path)
    if not pt_path.exists():
        if not build_if_missing: raise FileNotFoundError(f"{pt_path} not found.")
        data = build_hetero_from_primekg(save=True)
    else:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); data = torch.load(pt_path, weights_only=False)
    # Some cached graphs may miss num_nodes on isolated types; backfill defensively.
    for nt in data.node_types:
        if not hasattr(data[nt], "num_nodes") or data[nt].num_nodes is None:
            names = getattr(data[nt], "node_names", None)
            data[nt].num_nodes = len(names) if names is not None else 0
    if with_features:
        build_text_features(data, force_fallback=force_fallback_features)
    targets = {"indication": _find_target_edge(data, INDICATION_REL),
               "contraindication": _find_target_edge(data, CONTRAINDICATION_REL)}
    return data, targets

def graph_summary(data, targets):
    '''Return a multi-line human-readable summary of node types, edges, and targets.'''
    lines = ["PrimeKG graph:"]; total_nodes = total_edges = 0
    for nt in data.node_types:
        n = int(data[nt].num_nodes); total_nodes += n
        dim = int(data[nt].x.size(1)) if "x" in data[nt] else None
        extra = f" feat_dim={dim}" if dim else ""
        if nt == DISEASE_TYPE and "is_oncology" in data[nt]:
            extra += f" oncology={int(data[nt].is_oncology.sum())}"
        lines.append(f"  node {nt}: {n}{extra}")
    for et in data.edge_types: total_edges += int(data[et].edge_index.size(1))
    lines.append(f"  totals: {total_nodes:,} nodes, {total_edges:,} edges, {len(data.edge_types)} edge types")
    for name, et in targets.items():
        lines.append(f"  target [{name}]: {et} = {int(data[et].edge_index.size(1))} edges" if et
                     else f"  target [{name}]: NOT FOUND")
    return "\n".join(lines)
print("L3 ready")

### L4. The models

Four link-prediction models, all answering the same question: how likely is this (drug, disease) edge?

- `HeteroGNN` runs message passing over the heterogeneous graph. Each node repeatedly gathers information from its neighbors across every relation type (SAGEConv inside HeteroConv), which gives each node a context-aware embedding. An `EdgeMLPDecoder` then scores a pair from the two node embeddings. It also carries a `MechanismHead` for the recovery task in section 7, which scores a (drug, gene, disease) triple, in effect asking whether that gene is the mechanism bridge.
- `FeatureMLP` ignores the edges. It transforms each node's raw features with an MLP and scores pairs, which isolates what the features alone can do.
- `DistMultKGE` is a standard knowledge-graph embedding: one learned vector per drug and per disease plus a relation vector, scored with a bilinear product. No features and no message passing, just structural memorization.

Comparing the four shows where the predictive signal actually comes from.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv

class EdgeMLPDecoder(nn.Module):
    '''Score an edge from the concatenation of its endpoints' embeddings -> one logit.'''
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z_src, z_dst):
        return self.net(torch.cat([z_src, z_dst], dim=-1)).squeeze(-1)

class MechanismHead(nn.Module):
    '''Score a (drug, gene, disease) triple -> one logit ('is this gene the bridge?').'''
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z_drug, z_gene, z_dis):
        return self.net(torch.cat([z_drug, z_gene, z_dis], dim=-1)).squeeze(-1)

class HeteroGNN(nn.Module):
    '''Heterogeneous GraphSAGE encoder + edge decoder + mechanism head.'''
    def __init__(self, node_types, edge_types, in_dims, hidden=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.node_types = node_types; self.edge_types = edge_types; self.dropout = dropout
        # Project each node type's (possibly different-dim) features into a shared hidden space.
        self.proj = nn.ModuleDict({nt: nn.Linear(in_dims[nt], hidden) for nt in node_types})
        # Each layer applies one SAGEConv *per relation* and sums the messages per node.
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr="sum")
            for _ in range(num_layers)
        ])
        self.decoder = EdgeMLPDecoder(hidden); self.mech_head = MechanismHead(hidden)
    def encode(self, data):
        '''Run message passing and return a dict {node_type: embeddings}.'''
        device = next(self.parameters()).device
        x_dict = {nt: self.proj[nt](data[nt].x.to(device)) for nt in self.node_types if nt in data.node_types}
        eid = {et: data[et].edge_index.to(device) for et in self.edge_types
               if et in data.edge_types and data[et].edge_index.numel() > 0}
        for conv in self.convs:
            out = conv(x_dict, eid)
            for nt in x_dict:                      # node types that received no messages: keep their value
                if nt not in out: out[nt] = x_dict[nt]
            # nonlinearity + dropout between layers
            x_dict = {nt: F.dropout(F.relu(v), p=self.dropout, training=self.training) for nt, v in out.items()}
        return x_dict
    def decode(self, z, et, edge_label_index):
        '''Score a batch of candidate edges of type `et`.'''
        s_t, _, d_t = et; dev = z[s_t].device; eli = edge_label_index.to(dev)
        return self.decoder(z[s_t][eli[0]], z[d_t][eli[1]])
    def score_mechanism(self, z, drug_i, gene_i, dis_i, dt="drug", gt="gene_protein", ct="disease"):
        '''Score (drug, gene, disease) triples for the mechanism-recovery task.'''
        dev = z[dt].device
        return self.mech_head(z[dt][drug_i.to(dev)], z[gt][gene_i.to(dev)], z[ct][dis_i.to(dev)])

class FeatureMLP(nn.Module):
    '''Edges-ignored baseline: MLP on raw node features only (no message passing).'''
    def __init__(self, node_types, in_dims, hidden=128, dropout=0.3):
        super().__init__(); self.node_types = node_types; self.dropout = dropout
        self.proj = nn.ModuleDict({nt: nn.Linear(in_dims[nt], hidden) for nt in node_types})
        self.decoder = EdgeMLPDecoder(hidden)
    def encode(self, data):
        device = next(self.parameters()).device
        return {nt: F.relu(self.proj[nt](data[nt].x.to(device))) for nt in self.node_types if nt in data.node_types}
    def decode(self, z, et, edge_label_index):
        s_t, _, d_t = et; dev = z[s_t].device; eli = edge_label_index.to(dev)
        return self.decoder(z[s_t][eli[0]], z[d_t][eli[1]])

class DistMultKGE(nn.Module):
    '''DistMult knowledge-graph embedding: learn drug/disease vectors + a relation vector.'''
    def __init__(self, src_type, dst_type, num_src, num_dst, dim=128):
        super().__init__(); self.src_type, self.dst_type = src_type, dst_type
        self.src_emb = nn.Embedding(num_src, dim); self.dst_emb = nn.Embedding(num_dst, dim)
        self.rel = nn.Parameter(torch.randn(dim) * 0.1)        # single learned relation vector
        nn.init.xavier_uniform_(self.src_emb.weight); nn.init.xavier_uniform_(self.dst_emb.weight)
    def score(self, edge_label_index):
        dev = self.rel.device; eli = edge_label_index.to(dev)
        # DistMult score = <src, rel, dst> elementwise then summed.
        return (self.src_emb(eli[0]) * self.rel * self.dst_emb(eli[1])).sum(-1)
print("L4 ready")

### L5. Metrics

The usual link-prediction metrics, all computed from predicted scores against binary labels:

- AUROC: the chance a random positive outscores a random negative (0.5 is chance).
- AUPRC: area under precision-recall, more informative than AUROC when positives are rare.
- Hits@k and MRR: ranking quality, whether the true edge sits near the top.
- Precision, recall, F1 at the threshold that maximizes F1.

`compute_all_metrics` returns all of them at once and falls back to zeros for degenerate label sets (all positive or all negative).

In [ ]:
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

def _to_np(x):
    if isinstance(x, torch.Tensor): return x.detach().cpu().numpy()
    if isinstance(x, list): return np.array(x)
    return x

def hits_at_k(y, s, k=10):
    '''Fraction of true positives captured in the top-k highest-scoring items.'''
    y, s = _to_np(y), _to_np(s); order = np.argsort(-s)[:k]; total_pos = float(np.sum(y))
    return 0.0 if total_pos == 0 else float(np.sum(y[order]) / min(k, total_pos))

def mean_reciprocal_rank(y, s):
    '''Average of 1/rank over the positive items (higher = positives ranked nearer the top).'''
    y, s = _to_np(y), _to_np(s); ranked = y[np.argsort(-s)]; positions = np.where(ranked == 1)[0] + 1
    return 0.0 if positions.size == 0 else float(np.mean(1.0 / positions))

def optimal_threshold_metrics(y, s):
    '''Precision/recall/F1 at the score threshold that maximizes F1.'''
    prec, rec, thr = precision_recall_curve(y, s)
    f1 = 2*(prec[:-1]*rec[:-1]) / (prec[:-1]+rec[:-1]+1e-10)
    if f1.size == 0: return {"precision":0.0,"recall":0.0,"f1":0.0,"threshold":0.5}
    i = int(np.argmax(f1)); t = float(thr[i]); pred = (s >= t).astype(int)
    return {"precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "f1":float(f1_score(y,pred,zero_division=0)),"threshold":t}

def compute_all_metrics(y, s, prefix=""):
    '''One-stop metric dict. Returns zeros if labels are degenerate (all 0s or all 1s).'''
    y, s = _to_np(y), _to_np(s)
    keys = ["auroc","auprc","hits@1","hits@3","hits@10","mrr","precision","recall","f1"]
    if len(y) == 0 or np.sum(y) == 0 or np.sum(y) == len(y):
        return {f"{prefix}{k}":0.0 for k in keys}
    tm = optimal_threshold_metrics(y, s)
    return {f"{prefix}auroc":float(roc_auc_score(y,s)), f"{prefix}auprc":float(average_precision_score(y,s)),
            f"{prefix}hits@1":hits_at_k(y,s,1), f"{prefix}hits@3":hits_at_k(y,s,3),
            f"{prefix}hits@10":hits_at_k(y,s,10), f"{prefix}mrr":mean_reciprocal_rank(y,s),
            f"{prefix}precision":tm["precision"], f"{prefix}recall":tm["recall"], f"{prefix}f1":tm["f1"]}
print("L5 ready")

### L6. Leakage-safe splits

This is the part that decides whether the benchmark is honest. A naive edge split leaks the answer: the target edge would still be visible during message passing, and a held-out disease could in effect see its own indications. The splitter prevents that:

- `_build_base_graph` strips every drug-disease therapeutic edge from the message-passing graph and keeps only the training target edges, so the model never sees validation or test labels in the graph it builds embeddings from.
- Negatives are sampled as (drug, disease) pairs that are not known positives, so the model has to discriminate rather than memorize.

There are three regimes, in increasing difficulty:
1. Transductive: a random edge split, with every node seen during training.
2. Inductive cold-disease (oncology): whole diseases are held out, so the model has to generalize to cancers it never trained on. This is the realistic repurposing setting.
3. Inductive cold-drug: whole drugs are held out.

Two ablations probe what the GNN leans on. `ablate_topology` either rewires (shuffle) or deletes (empty) the non-target edges, and `drop_relations` removes specific edge families such as drug-protein.

In [ ]:
from dataclasses import dataclass, field
from typing import Set, Tuple, List, Optional

# Normalized therapeutic relation names, for fast membership checks.
_THERAPEUTIC_NORM = {_norm_rel(r) for r in THERAPEUTIC_RELS}

@dataclass
class SplitData:
    '''Bundles a leakage-safe message-passing graph (`base`) with train/val/test labels.'''
    base: HeteroData; target_edge_type: tuple; regime: str
    train_label_index: torch.Tensor; train_label: torch.Tensor
    val_label_index: torch.Tensor; val_label: torch.Tensor
    test_label_index: torch.Tensor; test_label: torch.Tensor
    info: dict = field(default_factory=dict)

def _therapeutic_edge_types(data):
    '''All drug<->disease therapeutic edge types (both directions) - stripped to avoid leakage.'''
    out = []
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r in _THERAPEUTIC_NORM: out.append(et)
    return out

def _sample_negatives(pos_set, src_pool, dst_pool, num, gen):
    '''Sample `num` (src, dst) pairs that are NOT in `pos_set` (negative training edges).'''
    if num <= 0 or src_pool.numel() == 0 or dst_pool.numel() == 0:
        return torch.empty((2,0), dtype=torch.long)
    neg_src, neg_dst, seen = [], [], set(); attempts = 0; max_attempts = max(2000, num*50)
    while len(neg_src) < num and attempts < max_attempts:
        batch = max(num*2, 128)                                  # over-sample, then filter
        si = src_pool[torch.randint(0, src_pool.numel(), (batch,), generator=gen)]
        di = dst_pool[torch.randint(0, dst_pool.numel(), (batch,), generator=gen)]
        for s, d in zip(si.tolist(), di.tolist()):
            key = (s, d)
            if key in pos_set or key in seen: continue          # skip true edges & duplicates
            seen.add(key); neg_src.append(s); neg_dst.append(d)
            if len(neg_src) >= num: break
        attempts += batch
    return torch.tensor([neg_src, neg_dst], dtype=torch.long)

def _build_base_graph(data, target_edge_type, train_target_cols):
    '''Copy the graph but: keep only TRAIN target edges, and delete ALL therapeutic edges.'''
    base = HeteroData()
    for nt in data.node_types:                                  # copy node stores verbatim
        for k, v in data[nt].items(): base[nt][k] = v
    therapeutic = set(_therapeutic_edge_types(data))
    for et in data.edge_types:
        if et == target_edge_type:                              # target: expose only training edges
            base[et].edge_index = data[et].edge_index[:, train_target_cols]
        elif et in therapeutic:                                 # other therapeutic edges: remove entirely
            base[et].edge_index = data[et].edge_index[:, :0]
        else:                                                   # all biology edges: keep
            base[et].edge_index = data[et].edge_index
    return base

def _make(base, target, regime, tr_pos, tr_neg, va_pos, va_neg, te_pos, te_neg, info):
    '''Assemble positives+negatives into labeled index/label tensors for each split.'''
    def cat(pos, neg):
        return (torch.cat([pos, neg], dim=1),
                torch.cat([torch.ones(pos.size(1)), torch.zeros(neg.size(1))]))
    tr_i, tr_l = cat(tr_pos, tr_neg); va_i, va_l = cat(va_pos, va_neg); te_i, te_l = cat(te_pos, te_neg)
    return SplitData(base, target, regime, tr_i, tr_l, va_i, va_l, te_i, te_l, info)

def transductive_split(data, target, val_frac=0.1, test_frac=0.2, neg_ratio=1.0, seed=0):
    '''Random edge split: shuffle target edges into train/val/test (all nodes seen).'''
    gen = torch.Generator().manual_seed(seed); s_t, _, d_t = target
    pos = data[target].edge_index; n = pos.size(1); perm = torch.randperm(n, generator=gen)
    n_test, n_val = int(round(test_frac*n)), int(round(val_frac*n))
    test_c, val_c, train_c = perm[:n_test], perm[n_test:n_test+n_val], perm[n_test+n_val:]
    tr_pos, va_pos, te_pos = pos[:, train_c], pos[:, val_c], pos[:, test_c]
    pos_set = {(int(s), int(d)) for s, d in zip(pos[0].tolist(), pos[1].tolist())}
    src_pool = torch.arange(int(data[s_t].num_nodes)); dst_pool = torch.arange(int(data[d_t].num_nodes))
    base = _build_base_graph(data, target, train_c)
    def neg(p): return _sample_negatives(pos_set, src_pool, dst_pool, int(round(p.size(1)*neg_ratio)), gen)
    info = {"regime":"transductive","train_pos":int(tr_pos.size(1)),
            "val_pos":int(va_pos.size(1)),"test_pos":int(te_pos.size(1))}
    return _make(base, target, "transductive", tr_pos, neg(tr_pos), va_pos, neg(va_pos), te_pos, neg(te_pos), info)

def inductive_node_split(data, target, holdout_side="dst", val_frac=0.1, test_frac=0.2,
                         neg_ratio=1.0, seed=0, restrict_oncology=False):
    '''Cold-node split: hold out whole drugs ('src') or whole diseases ('dst').

    Every edge touching a held-out node goes to val/test, so those nodes are never seen
    during training - the realistic 'generalize to a new disease/drug' setting.
    '''
    gen = torch.Generator().manual_seed(seed); s_t, _, d_t = target
    pos = data[target].edge_index
    side_row = 0 if holdout_side == "src" else 1
    side_type = s_t if holdout_side == "src" else d_t
    participating = torch.unique(pos[side_row])                  # nodes that have at least one target edge
    if restrict_oncology and holdout_side == "dst" and "is_oncology" in data[d_t]:
        onc = data[d_t].is_oncology                              # restrict held-out diseases to cancers
        participating = torch.tensor([n for n in participating.tolist() if bool(onc[n])], dtype=torch.long)
    perm = participating[torch.randperm(participating.numel(), generator=gen)]; n = participating.numel()
    n_test, n_val = max(1, int(round(test_frac*n))), max(1, int(round(val_frac*n)))
    test_nodes = set(perm[:n_test].tolist()); val_nodes = set(perm[n_test:n_test+n_val].tolist())
    held = test_nodes | val_nodes
    side_idx = pos[side_row].tolist(); tr_c, va_c, te_c = [], [], []
    for col, nd in enumerate(side_idx):                          # route each edge by its held-out endpoint
        (te_c if nd in test_nodes else va_c if nd in val_nodes else tr_c).append(col)
    tr_c = torch.tensor(tr_c, dtype=torch.long); tr_pos = pos[:, tr_c]
    va_pos = pos[:, torch.tensor(va_c, dtype=torch.long)] if va_c else pos[:, :0]
    te_pos = pos[:, torch.tensor(te_c, dtype=torch.long)] if te_c else pos[:, :0]
    pos_set = {(int(s), int(d)) for s, d in zip(pos[0].tolist(), pos[1].tolist())}
    other_type = d_t if holdout_side == "src" else s_t
    other_pool = torch.arange(int(data[other_type].num_nodes))
    train_side = torch.tensor(sorted(set(participating.tolist()) - held), dtype=torch.long)
    base = _build_base_graph(data, target, tr_c)
    def neg(p, side_pool):                                       # negatives must use the matching held-out nodes
        num = int(round(p.size(1)*neg_ratio))
        return (_sample_negatives(pos_set, side_pool, other_pool, num, gen) if holdout_side == "src"
                else _sample_negatives(pos_set, other_pool, side_pool, num, gen))
    def pool(s): return torch.tensor(sorted(s), dtype=torch.long)
    info = {"regime":f"inductive_cold_{holdout_side}","restrict_oncology":restrict_oncology,
            "n_test_nodes":len(test_nodes),"n_val_nodes":len(val_nodes),
            "train_pos":int(tr_pos.size(1)),"val_pos":int(va_pos.size(1)),"test_pos":int(te_pos.size(1))}
    return _make(base, target, info["regime"], tr_pos, neg(tr_pos, train_side),
                 va_pos, neg(va_pos, pool(val_nodes)), te_pos, neg(te_pos, pool(test_nodes)), info)

def make_split(data, target, regime, seed=0, holdout_side="dst", val_frac=0.1, test_frac=0.2,
               neg_ratio=1.0, restrict_oncology=False):
    '''Dispatch to the right split function by regime name.'''
    if regime == "transductive":
        return transductive_split(data, target, val_frac, test_frac, neg_ratio, seed)
    if regime in ("inductive","inductive_cold_dst","inductive_cold_src"):
        side = "src" if regime.endswith("src") else holdout_side
        return inductive_node_split(data, target, side, val_frac, test_frac, neg_ratio, seed, restrict_oncology)
    raise ValueError(regime)

def ablate_topology(split, mode, seed=0):
    '''Return a copy of the split with non-target edges 'shuffle'd (rewired) or 'empty'd (removed).'''
    gen = torch.Generator().manual_seed(seed); nb = HeteroData()
    for nt in split.base.node_types:
        for k, v in split.base[nt].items(): nb[nt][k] = v
    for et in split.base.edge_types:
        ei = split.base[et].edge_index
        if et == split.target_edge_type: nb[et].edge_index = ei; continue   # never touch target edges
        if mode == "empty":
            nb[et].edge_index = ei[:, :0]                                    # delete this relation
        elif mode == "shuffle":
            perm = torch.randperm(ei.size(1), generator=gen)                 # randomly rewire destinations
            nb[et].edge_index = torch.stack([ei[0], ei[1][perm]], 0)
        else:
            raise ValueError(mode)
    return SplitData(nb, split.target_edge_type, f"{split.regime}_ablate_{mode}", split.train_label_index,
                     split.train_label, split.val_label_index, split.val_label, split.test_label_index,
                     split.test_label, dict(split.info))

def drop_relations(split, drop_substrings):
    '''Return a copy of the split with edge families whose relation matches a substring removed.'''
    nb = HeteroData()
    for nt in split.base.node_types:
        for k, v in split.base[nt].items(): nb[nt][k] = v
    for et in split.base.edge_types:
        _, rel, _ = et
        if et != split.target_edge_type and any(sub in rel for sub in drop_substrings):
            continue                                                          # skip = drop this relation
        nb[et].edge_index = split.base[et].edge_index
    return SplitData(nb, split.target_edge_type, f"{split.regime}_drop", split.train_label_index,
                     split.train_label, split.val_label_index, split.val_label, split.test_label_index,
                     split.test_label, dict(split.info))
print("L6 ready")

### L7. Training loops

One loop per model family. They all optimize binary cross-entropy on positive and negative edges, track validation AUROC each epoch, and keep the best weights (early stopping), so the reported numbers do not come from an over-fit final epoch.

`train_gnn_joint` is the exception. It adds a second objective to the link loss: for each curated (drug, disease) pair it has to rank the true bridge gene above several degree-matched decoy genes, so it cannot win just by preferring popular genes. That is what lets the GNN do mechanism recovery later.

`set_all_seeds` fixes the RNGs for reproducibility, and `evaluate_model` reports metrics for any of the model types.

In [ ]:
import copy, random

def set_all_seeds(seed):
    '''Pin Python/NumPy/Torch RNGs so a run is reproducible.'''
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

@torch.no_grad()
def _eval_encoder(model, base, et, eli, labels, device):
    '''Encode the graph once, score the labeled edges, and compute metrics.'''
    model.eval(); z = model.encode(base); scores = torch.sigmoid(model.decode(z, et, eli)).cpu()
    return compute_all_metrics(labels.cpu(), scores)

def train_gnn(model, split, device, epochs=50, patience=10, lr=5e-3, weight_decay=1e-5):
    '''Train the HeteroGNN on link prediction with early stopping on val AUROC.'''
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        z = model.encode(split.base)                                       # message passing
        loss = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_encoder(model, split.base, et, split.val_label_index, split.val_label, device)["auroc"]
        if val > best:                                                     # remember the best checkpoint
            best, wait = val, 0
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break                                     # early stop
    if best_state is not None: model.load_state_dict(best_state)
    return model

def train_mlp(model, split, device, epochs=200, patience=20, lr=5e-3, weight_decay=1e-5):
    '''Train the features-only MLP baseline (same loop, no graph messages).'''
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        loss = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_encoder(model, split.base, et, split.val_label_index, split.val_label, device)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def _eval_kge(model, eli, labels):
    model.eval(); return compute_all_metrics(labels.cpu(), torch.sigmoid(model.score(eli)).cpu())

def train_kge(model, split, device, epochs=300, patience=30, lr=1e-2, weight_decay=1e-6):
    '''Train the DistMult KGE baseline (scores from learned embeddings only).'''
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model.score(split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_kge(model, split.val_label_index, split.val_label)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

def train_gnn_joint(model, split, device, mech, decoys, lam=1.0, n_neg=8, mech_batch=256,
                    epochs=60, lr=5e-3, weight_decay=1e-5):
    '''Joint training: link-prediction loss + contrastive mechanism (bridge-gene) loss.

    For each curated (drug, disease, bridge_gene) example we build a candidate set of
    [true gene + n_neg degree-matched decoys] and use cross-entropy to push the true
    gene's mechanism-score to the top. `lam` weights this objective against the link loss.
    '''
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device)
    md_, mc, mg = mech.drug, mech.dis, mech.gene; M = int(md_.numel())
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        link = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        mech_loss = torch.tensor(0.0, device=device)
        if M > 0 and lam > 0:
            b = min(mech_batch, M); idx = torch.randint(0, M, (b,))         # sample a batch of examples
            bd, bc, bg = md_[idx], mc[idx], mg[idx]
            # For each example draw n_neg decoy genes from the same degree bucket as the true gene.
            neg = torch.tensor([decoys.sample(int(bg[j]), {int(bg[j])}, n_neg) for j in range(b)], dtype=torch.long)
            cand = torch.cat([bg.view(-1,1), neg], dim=1); K = cand.size(1) # column 0 = the true gene
            fd = bd.view(-1,1).expand(-1,K).reshape(-1); fc = bc.view(-1,1).expand(-1,K).reshape(-1)
            fg = cand.reshape(-1)
            logits = model.score_mechanism(z, fd, fg, fc).view(b, K)
            # Cross-entropy with target 0 => "rank the true gene (col 0) above the decoys".
            mech_loss = F.cross_entropy(logits, torch.zeros(b, dtype=torch.long, device=logits.device))
        (link + lam*mech_loss).backward(); opt.step()
    return model

def evaluate_model(model, split, device, which="test"):
    '''Return metrics on the chosen split ('test' or 'val') for any model type.'''
    eli = getattr(split, f"{which}_label_index"); lab = getattr(split, f"{which}_label")
    if isinstance(model, DistMultKGE): return _eval_kge(model, eli, lab)
    return _eval_encoder(model, split.base, split.target_edge_type, eli, lab, device)
print("L7 ready")

---
# Workflow

## 2. PrimeKG: download, build, load, inspect

This downloads PrimeKG once, builds or loads the graph, attaches node features (hashing in fast mode, SentenceTransformer in full mode), and prints a summary plus a per-type table. Afterwards `data` holds the graph and `target` is the `(drug, indication, disease)` edge type we predict.

In [ ]:
download_primekg()                               # one-time ~980 MB download (skipped if cached)
if not HETERODATA_PT.exists():
    build_hetero_from_primekg()                  # parse CSV -> HeteroData (cached)
data, targets = load_primekg(with_features=True, force_fallback_features=USE_FALLBACK_FEATS)
target = targets["indication"]                   # our main prediction target edge type
print(graph_summary(data, targets))

In [ ]:
# A quick per-node-type table to show the scale and feature dimension of each type.
import pandas as pd
rows = [{"node_type":nt,
         "num_nodes":int(data[nt].num_nodes),
         "feature_dim":int(data[nt].x.size(1)) if "x" in data[nt] else None,
         "example_names":", ".join(str(n) for n in list(getattr(data[nt],"node_names",[]))[:2])}
        for nt in data.node_types]
display(pd.DataFrame(rows).sort_values("num_nodes", ascending=False).reset_index(drop=True))
print("oncology diseases:", int(data[DISEASE_TYPE].is_oncology.sum()),
      "| indication edges:", int(data[target].edge_index.size(1)))

## 12. Finding 8: a popularity-deconfounded orthogonal check

The repurposing shortlist (Section 8) raises an obvious question: do the model's novel
predictions show up as real human trials? A naive check (do top-scored novel pairs have
ClinicalTrials.gov entries more than random pairs) is badly confounded by popularity on two
sides. Some drugs are trialed for every cancer, and some cancers are trialed with every drug,
so a positive result can mean "popular drug" or "popular cancer" rather than "right drug for
this cancer". We remove both effects and test only what is left: the drug $\times$ cancer
interaction.

We fit an additive two-way model, $\mathrm{score}(d,c)=\mu+\alpha_d+\beta_c+\varepsilon_{dc}$,
subtract the drug effect $\alpha_d$ and the cancer effect $\beta_c$, and ask whether the
residual $\varepsilon_{dc}$ still predicts a real trial. We also report the simpler
within-drug AUROC, which removes drug popularity only.

What we recorded on the full run (about 100 drugs by 18 cancers, roughly 1{,}800 pairs): the
within-drug AUROC is a strong-looking 0.821 ($p=5\times10^{-4}$), but once the cancer effect
is also removed the interaction AUROC is 0.475 ($p=0.85$), that is, no signal. The apparent
effect was cancer popularity (cervical cancer, for instance, scores about 0.998 for almost
every drug and is widely trialed), not pairwise specificity. This is an honest, fully
characterized negative, and it stands in deliberate contrast to the prospective result
(Findings 5 and 7), where specific predictive signal does appear.

This cell makes live ClinicalTrials.gov calls (cached to disk). In fast mode it uses only a
handful of drugs and cancers, so the live numbers are noisy; the recorded full-run numbers
above are the ones to read.

In [ ]:
# --- Finding 8: popularity-DECONFOUNDED orthogonal validation -------------------
import numpy as np, urllib.parse, urllib.request, json as _json, time, re as _re
from pathlib import Path as _Path
from sklearn.metrics import roc_auc_score

# Ranking drugs by name needs real semantic features (the fast-mode hashing features
# make the top-scored "drugs" arbitrary chemicals with no trials). Reuse Finding 5's
# ST-feature graph if present, else load it here so this part stands alone.
try:
    data_v, target_v = data_st, target_st
except NameError:
    data_v, _targets_v = load_primekg(with_features=True, force_fallback_features=False)
    target_v = _targets_v["indication"]

# A small bounded demo (live ClinicalTrials.gov). The recorded full run uses ~100
# drugs x 18 cancers; here we keep it tiny so the notebook finishes quickly.
DEMO_CANCERS = ["glioblastoma", "melanoma", "breast carcinoma", "ovarian cancer",
                "prostate cancer", "colorectal cancer"]
N_FOCUS  = 8 if FAST_MODE else 40
CAP_DRUGS = 12 if FAST_MODE else 80

# 1) Train a transductive deployment GNN on indication edges and score drugs x cancers.
set_all_seeds(0)
sp = make_split(data_v, target_v, "transductive", seed=0, val_frac=0.1, test_frac=0.0)
in_dims_v = {nt: int(data_v[nt].x.size(1)) for nt in data_v.node_types}
dep = HeteroGNN(list(data_v.node_types), list(sp.base.edge_types), in_dims_v).to(DEVICE)
dep = train_gnn(dep, sp, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE)

dnames_v  = list(data_v[DISEASE_TYPE].node_names)
rxnames_v = list(data_v[DRUG_TYPE].node_names)
onc = data_v[DISEASE_TYPE].is_oncology
name2dz = {}
for q in DEMO_CANCERS:
    for i, nm in enumerate(dnames_v):
        if q in str(nm).lower() and bool(onc[i]):
            name2dz[q] = i
            break
dz_ids = list(dict.fromkeys(name2dz.values()))

def _known_dd(data):
    known = set()
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and any(k in r for k in ("indication", "contra", "off-label")):
            ei = data[et].edge_index
            for a, b in zip(ei[0].tolist(), ei[1].tolist()):
                known.add((a, b) if s == DRUG_TYPE else (b, a))
    return known
known = _known_dd(data_v)

@torch.no_grad()
def score_all(model, dz):
    z = model.encode(sp.base)
    nd = int(data_v[DRUG_TYPE].num_nodes)
    eli = torch.stack([torch.arange(nd, device=z[DRUG_TYPE].device),
                       torch.full((nd,), dz, device=z[DRUG_TYPE].device)])
    return torch.sigmoid(model.decode(z, target_v, eli)).cpu().numpy()

scores_by_dz = {dz: score_all(dep, dz) for dz in dz_ids}

# focus drugs = union of each cancer's top-N novel (not-already-indicated) drugs
focus, seen = [], set()
for dz in dz_ids:
    cnt = 0
    for d in np.argsort(-scores_by_dz[dz]):
        d = int(d)
        if (d, dz) in known:
            continue
        if d not in seen:
            seen.add(d); focus.append(d)
        cnt += 1
        if cnt >= N_FOCUS:
            break
focus = focus[:CAP_DRUGS]

# 2) Query ClinicalTrials.gov (cached) for each (focus drug, demo cancer) novel pair.
CT_CACHE = _Path("data/clinicaltrials_cache.json")
ctc = _json.loads(CT_CACHE.read_text()) if CT_CACHE.exists() else {}

def ct_hit(drug, cond):
    key = drug.strip().lower() + "|||" + cond.strip().lower()
    if key in ctc and "error" not in ctc[key]:
        return int(ctc[key]["hit"])
    qd = _re.sub(r"[^A-Za-z0-9 -]", " ", str(drug)).strip()
    qc = _re.sub(r"[^A-Za-z0-9 -]", " ", str(cond)).strip()
    if not qd or not qc:
        ctc[key] = {"hit": 0, "total": 0}; return 0
    params = {"query.intr": qd, "query.cond": qc,
              "filter.advanced": "AREA[StudyType]INTERVENTIONAL",
              "countTotal": "true", "pageSize": "1", "format": "json"}
    url = "https://clinicaltrials.gov/api/v2/studies?" + urllib.parse.urlencode(params)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "oncoevidence/1.0"})
        with urllib.request.urlopen(req, timeout=30) as r:
            tot = int(_json.load(r).get("totalCount", 0) or 0)
        ctc[key] = {"hit": int(tot > 0), "total": tot}
    except Exception:
        return 0
    time.sleep(0.34)
    return ctc[key]["hit"]

dz_clean = {dz: str(dnames_v[dz]).replace(" (disease)", "") for dz in dz_ids}
S = np.full((len(focus), len(dz_ids)), np.nan)
H = np.full_like(S, np.nan)
for i, d in enumerate(focus):
    for j, dz in enumerate(dz_ids):
        if (d, dz) in known:
            continue
        S[i, j] = scores_by_dz[dz][d]
        H[i, j] = ct_hit(rxnames_v[d], dz_clean[dz])
CT_CACHE.parent.mkdir(exist_ok=True)
CT_CACHE.write_text(_json.dumps(ctc))
mask = ~np.isnan(S)

# 3) Remove drug + cancer popularity (two-way fixed effects); test the residual.
mu = float(np.nanmean(S)); a = np.zeros(S.shape[0]); b = np.zeros(S.shape[1])
Sc = np.where(mask, S, np.nan)
for _ in range(200):
    a = np.nan_to_num(np.nanmean(Sc - mu - b[None, :], axis=1))
    b = np.nan_to_num(np.nanmean(Sc - mu - a[:, None], axis=0))
R = Sc - mu - a[:, None] - b[None, :]
r = R[mask]; h = H[mask].astype(int)

conc = comp = 0.0   # within-drug AUROC (controls drug popularity only)
for i in range(S.shape[0]):
    sr = S[i, mask[i]]; hr = H[i, mask[i]].astype(int)
    pos = sr[hr == 1]; neg = sr[hr == 0]
    for sp_ in pos:
        comp += len(neg); conc += np.sum(sp_ > neg) + 0.5 * np.sum(sp_ == neg)
within = conc / comp if comp else float("nan")
inter = roc_auc_score(h, r) if 0 < h.sum() < len(h) else float("nan")

print(f"demo pairs={int(mask.sum())}  trial-hit fraction={np.nanmean(H):.3f}")
print(f"within-drug AUROC (drug popularity removed)  = {within:.3f}")
print(f"interaction AUROC (drug AND cancer removed)   = {inter:.3f}")
print("\n(recorded full run: within-drug 0.821 [p=5e-4] but interaction 0.475 [p=0.85] -->")
print(" the apparent signal is cancer popularity, not pairwise specificity: an honest negative.)")

## Recap and next: Part 9

Parts 1-8 establish the honest benchmark, the mechanism paths, the evidence layer, blinded recovery, the prospective audit, faithfulness, named cases, and the popularity-deconfounded negative. **Part 9** stress-tests whether the model ranks for the *right reason* (four counterfactuals), and **Part 10** adds independent functional-genomics corroboration (DepMap/GTEx).